# Run MoneyPrinterTurbo on Google Colab

This notebook installs the active `feat/solo-stack-presenter-quality-20261124` branch in an isolated Python 3.11 environment and launches its WebUI. Generated task folders and task-state records are stored in Google Drive so completed outputs and task history survive Colab runtime resets.


## 1. Install MoneyPrinterTurbo

The setup is safe to run again: it clones the repository on the first run and updates it on later runs. Dependencies are installed in the project's virtual environment to avoid conflicts with Colab's preinstalled packages.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/MoneyPrinterTurbo")
REPO_URL = "https://github.com/methaneex-oss/MoneyPrinterTurbo.git"
REPO_BRANCH = "feat/solo-stack-presenter-quality-20261124"

# Always install the active feature branch; main does not yet contain this work.
if (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH],
        check=True,
    )
    local_branch_exists = subprocess.run(
        ["git", "-C", str(REPO_DIR), "show-ref", "--verify", "--quiet",
         f"refs/heads/{REPO_BRANCH}"],
        check=False,
    ).returncode == 0
    if local_branch_exists:
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH],
            check=True,
        )
    else:
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "checkout", "-b", REPO_BRANCH,
             f"origin/{REPO_BRANCH}"],
            check=True,
        )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH],
        check=True,
    )
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository")
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--single-branch", "--branch",
         REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(REPO_DIR)
subprocess.run(["python", "-m", "pip", "install", "-q", "uv", "pyngrok"], check=True)
subprocess.run(["uv", "python", "install", "3.11"], check=True)
# Use the lockfile in an isolated environment without changing Colab's preinstalled packages.
subprocess.run(["uv", "sync", "--frozen", "--python", "3.11"], check=True)
print(f"MoneyPrinterTurbo is ready in {REPO_DIR}")

## Solo Stack: genuine official product visuals

A headless browser captures public product imagery from allowlisted official vendor pages when the episode actually names those products: ChatGPT, Claude, Notion, Canva, Grammarly, Zapier, Perplexity, Cursor, ClickUp, Figma, Gemini, Google Workspace, Microsoft 365, Asana, Trello, Airtable, Slack, Make, n8n, Loom, Descript, Calendly, HubSpot and GitHub Copilot. It does not log into accounts or fabricate application interfaces. Captured visuals are source-labelled and inserted into the narration-ordered timeline.


In [ ]:
import subprocess
from pathlib import Path

REPO_DIR = Path('/content/MoneyPrinterTurbo')
PROJECT_PYTHON = REPO_DIR / '.venv' / 'bin' / 'python'

# Install into MoneyPrinterTurbo's isolated environment; no paid service or account login.
subprocess.run(['uv', 'pip', 'install', '--python', str(PROJECT_PYTHON), 'playwright'], cwd=REPO_DIR, check=True)
subprocess.run([str(PROJECT_PYTHON), '-m', 'playwright', 'install', '--with-deps', 'chromium'], cwd=REPO_DIR, check=True)
print('Solo Stack official product visual capture is ready.')


In [ ]:
# Keep generated task folders in Google Drive so Colab runtime resets do not erase finished videos.
from google.colab import drive
from pathlib import Path
import os
import filecmp
import shutil

drive.mount('/content/drive')

REPO_DIR = Path('/content/MoneyPrinterTurbo')
MPT_HOME = Path('/content/drive/MyDrive/MoneyPrinterTurbo')
MPT_HOME.mkdir(parents=True, exist_ok=True)
os.environ['MPT_HOME'] = str(MPT_HOME)
TASKS_DIR = REPO_DIR / 'storage' / 'tasks'
DRIVE_TASKS_DIR = MPT_HOME / 'storage' / 'tasks'
DRIVE_TASKS_DIR.mkdir(parents=True, exist_ok=True)
TASKS_DIR.parent.mkdir(parents=True, exist_ok=True)

def merge_runtime_item(source: Path, destination: Path) -> None:
    """Merge without discarding either side when an existing file differs."""
    if source.is_dir():
        if destination.exists() and not destination.is_dir():
            raise RuntimeError(f'Cannot merge directory over a file: {destination}')
        destination.mkdir(parents=True, exist_ok=True)
        for child in source.iterdir():
            merge_runtime_item(child, destination / child.name)
        return

    if not source.is_file():
        raise RuntimeError(f'Unsupported task artifact type: {source}')

    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        if not destination.is_file():
            raise RuntimeError(f'Cannot merge file over a directory: {destination}')
        if filecmp.cmp(source, destination, shallow=False):
            return

        # Keep the existing Drive version before the current runtime version
        # replaces it. This also makes a partially completed prior migration safe.
        backup = destination.with_name(destination.name + '.drive-backup')
        suffix = 1
        while backup.exists():
            backup = destination.with_name(
                destination.name + f'.drive-backup-{suffix}'
            )
            suffix += 1
        shutil.copy2(destination, backup)

    shutil.copy2(source, destination)


# Migrate any task outputs already in this runtime before linking the folders.
if TASKS_DIR.is_symlink():
    current_target = TASKS_DIR.resolve()
    if current_target != DRIVE_TASKS_DIR.resolve():
        raise RuntimeError(f'Unexpected tasks symlink target: {current_target}')
elif TASKS_DIR.exists():
    for item in TASKS_DIR.iterdir():
        merge_runtime_item(item, DRIVE_TASKS_DIR / item.name)

    # Verify every local file against its Drive copy before deleting anything.
    for source_file in TASKS_DIR.rglob('*'):
        if source_file.is_file():
            destination = DRIVE_TASKS_DIR / source_file.relative_to(TASKS_DIR)
            if not destination.is_file() or not filecmp.cmp(
                source_file, destination, shallow=False
            ):
                raise RuntimeError(
                    f'Task migration verification failed: {source_file}'
                )

    shutil.rmtree(TASKS_DIR)
    TASKS_DIR.symlink_to(DRIVE_TASKS_DIR, target_is_directory=True)
else:
    TASKS_DIR.symlink_to(DRIVE_TASKS_DIR, target_is_directory=True)

print(f'Generated videos and task files will be saved in: {DRIVE_TASKS_DIR}')

In [ ]:
import subprocess
from pathlib import Path

REPO_DIR = Path('/content/MoneyPrinterTurbo')
CONFIG_PATH = Path(os.environ['MPT_HOME']) / 'config.toml'

# Create the standard config only on first run. Existing API keys and preferences are preserved.
if not CONFIG_PATH.exists():
    import shutil
    shutil.copy2(REPO_DIR / 'config.example.toml', CONFIG_PATH)

# Apply the documentary preset once. Later notebook runs preserve any settings you change in the WebUI.
preset_script = r'''
import os
from pathlib import Path
import toml

config_path = Path(os.environ.get('MPT_HOME', '.')) / 'config.toml'
config = toml.load(config_path)
app = config.setdefault('app', {})
ui = config.setdefault('ui', {})
preset_version = int(app.get('visual_storytelling_preset_version', 0) or 0)

if preset_version < 1:
    app['match_materials_to_script'] = True
    app['solo_stack_mode'] = True
    app['solo_stack_presenter_enabled'] = True
    app['visual_storytelling_preset_version'] = 1
    ui.update({
        'video_concat_mode': 'sequential',
        'video_transition_mode': 'FadeIn',
        'video_aspect_pixabay': '9:16',
        'video_fit_mode': 'cover',
        'video_clip_duration': 4,
        'video_clip_speed': 1.0,
        'bgm_volume': 0.1,
        'subtitle_position': 'two_thirds_bottom',
        'subtitle_display_mode': 'sentence',
        'subtitle_background_enabled': True,
        'subtitle_background_color': '#000000',
        'rounded_subtitle_background': True,
    })
    with config_path.open('w', encoding='utf-8') as handle:
        toml.dump(config, handle)
    print('Documentary editing preset applied. Existing API keys were preserved.')
else:
    print('Documentary editing preset already applied; current WebUI preferences were left unchanged.')

# Persist task status metadata as well as task folders across Colab runtime resets.
# This path is on the already-mounted Google Drive and is enabled by FileState.
app['task_state_dir'] = '/content/drive/MyDrive/MoneyPrinterTurbo/task-state'
app['enable_redis'] = False
app['solo_stack_mode'] = True
app['solo_stack_free_only'] = True
app['solo_stack_presenter_enabled'] = True
app['solo_stack_voice_performance_enabled'] = True
app['visual_storytelling_preset_version'] = max(preset_version, 4)
with config_path.open('w', encoding='utf-8') as handle:
    toml.dump(config, handle)
print('Persistent task state is configured on Google Drive.')
'''

subprocess.run(['uv', 'run', 'python', '-c', preset_script], cwd=REPO_DIR, check=True)


## 2. Enable free local AI scene images (GPU runtimes only)

This step selects a local image model only when the assigned Colab GPU and the model's usage terms permit the intended workflow. With at least 14 GiB VRAM it uses RealVisXL SDXL Lightning (OpenRAIL++). Between 8 and 14 GiB it falls back to Pixabay by default. SD-Turbo is an optional lower-VRAM model, but commercial use requires registration under Stability AI's Community License; its opt-in remains false until that registration is confirmed.

Model weights are downloaded on first warm-up and cached in Google Drive under `MPT_HOME/model-cache/huggingface`, so subsequent Colab sessions can reuse them. The local models generate still images; MoneyPrinterTurbo then adds camera motion and its scene-graphic system. This is not AI text-to-video. No paid image API is called.

In [ ]:
import os
from pathlib import Path
import json
import subprocess
import time
from urllib.error import URLError
from urllib.request import Request, urlopen

REPO_DIR = Path("/content/MoneyPrinterTurbo")
IMAGE_SERVER_DIR = REPO_DIR / "docs" / "local-image-server"
IMAGE_ENV = REPO_DIR / ".venv-image"
IMAGE_PYTHON = IMAGE_ENV / "bin" / "python"
IMAGE_LOG_PATH = Path("/content/mpt-image-server.log")
IMAGE_PORT = 7861
MODEL_CACHE = Path(os.environ["MPT_HOME"]) / "model-cache" / "huggingface"
MODEL_CACHE.mkdir(parents=True, exist_ok=True)

def stop_image_server():
    process = globals().get("image_server_proc")
    if process is not None and process.poll() is None:
        process.terminate()
        process.wait(timeout=15)
    log_handle = globals().get("image_server_log")
    if log_handle is not None and not log_handle.closed:
        log_handle.close()

def set_video_source(source, base_url="", model="", size=""):
    script = f"""
import os
import toml
from pathlib import Path
import tempfile
root = Path(os.environ.get("MPT_HOME", str(Path.cwd()))).expanduser().resolve()
path = root / "config.toml"
if not path.is_file():
    raise FileNotFoundError(f"Persistent MoneyPrinterTurbo config is missing: {{path}}")
cfg = toml.load(path)
app = cfg.setdefault("app", {{}})
app["video_source"] = {source!r}
app["allow_unverified_animatediff"] = {source == "animatediff"!r}
app["solo_stack_local_image_ready"] = {source == "openai_image"!r}
if {source!r} != "animatediff":
    app.pop("animatediff_base_url", None)
if {source!r} == "openai_image":
    app["openai_image_base_url"] = {base_url!r}
    app["openai_image_model"] = {model!r}
    app["openai_image_size"] = {size!r}
    app["openai_image_api_keys"] = []
if {source!r} == "animatediff":
    app["animatediff_base_url"] = "http://127.0.0.1:7862"
temporary_path = None
try:
    with tempfile.NamedTemporaryFile(
        "w", encoding="utf-8", dir=root, prefix=".config-", suffix=".toml", delete=False
    ) as handle:
        temporary_path = Path(handle.name)
        toml.dump(cfg, handle)
    os.replace(temporary_path, path)
finally:
    if temporary_path is not None:
        temporary_path.unlink(missing_ok=True)
"""
    subprocess.run(["uv", "run", "python", "-c", script], cwd=REPO_DIR, check=True)

def use_pixabay(reason):
    stop_image_server()
    set_video_source("pixabay")
    print(f"Local AI images unavailable ({reason}). Using the configured Pixabay source.")

stop_image_server()
# Clear stale local-server readiness from a previous Colab runtime.
set_video_source("pixabay")

try:
    import torch

    gpu_ready = torch.cuda.is_available()
    gpu_memory_gib = (
        torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        if gpu_ready else 0
    )
    print(f"CUDA available: {gpu_ready}; GPU memory: {gpu_memory_gib:.1f} GiB")

    SD_TURBO_COMMERCIAL_REGISTERED = False
    if not gpu_ready or gpu_memory_gib < 8:
        use_pixabay("Colab did not provide a CUDA GPU with at least 8 GiB VRAM")
    elif gpu_memory_gib >= 14:
        image_backend = "sdxl"
        image_model = "SG161222/RealVisXL_V5.0_Lightning"
        image_size = "832x1216"
    elif SD_TURBO_COMMERCIAL_REGISTERED:
        # SD-Turbo commercial production requires registration under the
        # Stability AI Community License. Keep this opt-in false by default.
        image_backend = "sd-turbo"
        image_model = "stabilityai/sd-turbo"
        image_size = "512x768"
    else:
        use_pixabay(
            "SD-Turbo is not enabled until Stability AI commercial registration is confirmed"
        )
    if gpu_ready and (
        gpu_memory_gib >= 14
        or (gpu_memory_gib >= 8 and SD_TURBO_COMMERCIAL_REGISTERED)
    ):
        print(f"Selected local image model: {image_backend} ({image_model})")
        subprocess.run(
            ["uv", "venv", "--python", "3.11", "--system-site-packages", str(IMAGE_ENV)],
            cwd=REPO_DIR,
            check=True,
        )
        subprocess.run(
            [
                "uv", "pip", "install", "--python", str(IMAGE_PYTHON),
                "-r", str(IMAGE_SERVER_DIR / "requirements.txt"),
            ],
            cwd=REPO_DIR,
            check=True,
        )

        image_server_log = IMAGE_LOG_PATH.open("w", encoding="utf-8")
        image_server_env = os.environ.copy()
        image_server_env["IMAGE_BACKEND"] = image_backend
        image_server_env["IMAGE_MODEL"] = image_model
        image_server_env["IMAGE_STEPS"] = "6" if image_backend == "sdxl" else "1"
        image_server_env["IMAGE_GUIDANCE"] = "1.5" if image_backend == "sdxl" else "0.0"
        image_server_env["HF_HOME"] = str(MODEL_CACHE)
        image_server_env["HUGGINGFACE_HUB_CACHE"] = str(MODEL_CACHE / "hub")
        image_server_proc = subprocess.Popen(
            [str(IMAGE_PYTHON), "server.py"],
            cwd=IMAGE_SERVER_DIR,
            env=image_server_env,
            stdout=image_server_log,
            stderr=subprocess.STDOUT,
            text=True,
        )

        deadline = time.time() + 90
        image_server_ready = False
        while time.time() < deadline:
            if image_server_proc.poll() is not None:
                break
            try:
                with urlopen(f"http://127.0.0.1:{IMAGE_PORT}/health", timeout=2) as response:
                    image_server_ready = response.status == 200
            except (URLError, TimeoutError):
                pass
            if image_server_ready:
                break
            time.sleep(2)

        if not image_server_ready:
            raise RuntimeError("local image server did not become healthy")

        # Load the weights now, before the WebUI starts. This prevents the
        # first generation request from timing out while downloading 6.6 GB.
        warmup_request = Request(
            f"http://127.0.0.1:{IMAGE_PORT}/warmup",
            data=b"",
            method="POST",
        )
        with urlopen(warmup_request, timeout=900) as response:
            warmup_result = json.loads(response.read().decode("utf-8"))
        if (
            not warmup_result.get("ok")
            or not warmup_result.get("on_gpu")
            or warmup_result.get("backend") != image_backend
            or warmup_result.get("model") != image_model
        ):
            raise RuntimeError(f"local image model warmup was not successful: {warmup_result}")

        set_video_source(
            "openai_image",
            f"http://127.0.0.1:{IMAGE_PORT}/v1",
            image_model,
            image_size,
        )
        print(
            f"Local {image_backend} image generation is ready. "
            "No image API key or paid provider is used."
        )

except Exception as error:
    try:
        use_pixabay(f"{type(error).__name__}: {error}")
    except Exception as config_error:
        print(f"Could not apply the Pixabay fallback: {config_error}")
    print("The WebUI can still launch using the configured stock-video workflow.");


## 3. Experimental local AnimateDiff video engine — disabled by default

The repository contains an isolated AnimateDiff text-to-video backend, but it is not part of the default Solo Stack production path. The adapter's authoritative commercial-use licence remains unresolved, so the notebook deliberately does not download its weights, stop the local image model, or switch the video source to AnimateDiff. This avoids putting a potentially non-commercial model into a public channel workflow.

The code remains available for a separately authorized non-commercial experiment. Its explicit opt-in constant defaults to `False`. Do not enable it for monetized production unless the adapter's authoritative weight licence has been located and reviewed. The LTX-Video 2B candidate also remains disabled pending its own model-weight licence and runtime validation.

The normal production path remains local SDXL/SD-Turbo still-image generation when the Colab GPU supports it, with the configured Pixabay source as the no-cost fallback. These paths do not make paid inference calls.

In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path
from urllib.error import URLError
from urllib.request import Request, urlopen

REPO_DIR = Path('/content/MoneyPrinterTurbo')
VIDEO_ENV = Path('/content/.venv-mpt-video')
VIDEO_PYTHON = VIDEO_ENV / 'bin' / 'python'
VIDEO_REQUIREMENTS = REPO_DIR / 'docs/local-video-models/requirements.txt'
VIDEO_SERVER = REPO_DIR / 'docs/local-video-models/server.py'
DRIVE_ROOT = Path('/content/drive/MyDrive/MoneyPrinterTurbo')
MODEL_CACHE = DRIVE_ROOT / 'model-cache' / 'huggingface'
VIDEO_LOG_PATH = Path('/content/mpt-animatediff-server.log')
VIDEO_PORT = 7862
VIDEO_HEALTH_URL = f'http://127.0.0.1:{VIDEO_PORT}/health'
image_server_stopped = False


def stop_video_server():
    process = globals().get('video_server_proc')
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=10)
    handle = globals().get('video_server_log')
    if handle is not None and not handle.closed:
        handle.close()


def restore_local_image_source():
    """Restart the local image model after video setup fails and VRAM is released."""
    global image_server_proc, image_server_log
    image_env = globals().get("image_server_env")
    if not image_server_stopped or not image_env:
        return False
    try:
        image_server_log = IMAGE_LOG_PATH.open("a", encoding="utf-8")
        image_server_proc = subprocess.Popen(
            [str(IMAGE_PYTHON), "server.py"],
            cwd=IMAGE_SERVER_DIR,
            env=image_env,
            stdout=image_server_log,
            stderr=subprocess.STDOUT,
            text=True,
        )
        deadline = time.time() + 120
        ready = False
        while time.time() < deadline:
            if image_server_proc.poll() is not None:
                break
            try:
                with urlopen(f"http://127.0.0.1:{IMAGE_PORT}/health", timeout=2) as response:
                    ready = response.status == 200
            except (URLError, TimeoutError):
                pass
            if ready:
                break
            time.sleep(2)
        if not ready:
            stop_image_server()
            return False
        warmup_request = Request(
            f"http://127.0.0.1:{IMAGE_PORT}/warmup",
            data=b"",
            method="POST",
        )
        with urlopen(warmup_request, timeout=900) as response:
            import json
            warmup_result = json.loads(response.read().decode("utf-8"))
        if (
            not warmup_result.get("ok")
            or not warmup_result.get("on_gpu")
            or warmup_result.get("model") != image_model
        ):
            stop_image_server()
            return False
        set_video_source(
            "openai_image",
            f"http://127.0.0.1:{IMAGE_PORT}/v1",
            image_model,
            image_size,
        )
        return True
    except Exception as restore_error:
        print(f"Local image model could not be restored: {type(restore_error).__name__}: {restore_error}")
        stop_image_server()
        return False


def use_existing_visual_source(reason):
    stop_video_server()
    if image_server_stopped:
        if not restore_local_image_source():
            set_video_source("pixabay")
            print("Restored the configured Pixabay source as the no-cost fallback.")
        else:
            print("Restored the local AI image model as the no-cost fallback.")
    print(f'AnimateDiff not enabled: {reason}')
    print('No paid inference API is called. No paid fallback is configured.')



# Keep the uncertain motion-adapter licence out of the default public-channel
# workflow. Enable only for a separately approved non-commercial experiment.
ALLOW_ANIMATEDIFF_EXPERIMENTAL = False
if not ALLOW_ANIMATEDIFF_EXPERIMENTAL:
    use_existing_visual_source(
        "the motion-adapter commercial-use licence is unresolved; "
        "AnimateDiff is disabled by default"
    )
else:
    stop_video_server()
    if not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError('Google Drive must be mounted by the task-persistence cell above.')
    if not VIDEO_SERVER.is_file() or not VIDEO_REQUIREMENTS.is_file():
        raise RuntimeError('Local AnimateDiff server files are missing from this repository branch.')

    try:
        import torch

        gpu_ready = torch.cuda.is_available()
        gpu_name = torch.cuda.get_device_name(0) if gpu_ready else 'none'
        vram_gib = (
            torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
            if gpu_ready else 0.0
        )
        free_disk_gib = shutil.disk_usage('/content').free / (1024 ** 3)
        ram_gib = (
            os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES')
        ) / (1024 ** 3)
        print({
            'gpu': gpu_name,
            'vram_gib': round(vram_gib, 2),
            'ram_gib': round(ram_gib, 2),
            'free_disk_gib': round(free_disk_gib, 2),
        })

        if not gpu_ready or vram_gib < 12:
            use_existing_visual_source(
                'a CUDA GPU with at least 12 GiB VRAM is required'
            )
        elif ram_gib < 10 or free_disk_gib < 16:
            use_existing_visual_source(
                'at least 10 GiB system RAM and 16 GiB free runtime disk are required'
            )
        else:
            # The image-generation service also occupies VRAM. Stop it only after
            # confirming this runtime can attempt video generation.
            stop_image_server()
            image_server_stopped = True
            torch.cuda.empty_cache()

            MODEL_CACHE.mkdir(parents=True, exist_ok=True)
            video_env = os.environ.copy()
            video_env['HF_HOME'] = str(MODEL_CACHE)
            video_env['HUGGINGFACE_HUB_CACHE'] = str(MODEL_CACHE / 'hub')
            video_env['ANIMATEDIFF_PORT'] = str(VIDEO_PORT)
            video_env['ANIMATEDIFF_MIN_VRAM_GIB'] = '12'

            if not VIDEO_PYTHON.exists():
                subprocess.run(
                    [
                        sys.executable, '-m', 'venv',
                        '--system-site-packages', str(VIDEO_ENV),
                    ],
                    check=True,
                )
            subprocess.run(
                [
                    str(VIDEO_PYTHON), '-m', 'pip', 'install',
                    '--disable-pip-version-check', '-r', str(VIDEO_REQUIREMENTS),
                ],
                check=True,
                env=video_env,
            )

            video_server_log = VIDEO_LOG_PATH.open('w', encoding='utf-8')
            video_server_proc = subprocess.Popen(
                [str(VIDEO_PYTHON), str(VIDEO_SERVER)],
                cwd=VIDEO_SERVER.parent,
                env=video_env,
                stdout=video_server_log,
                stderr=subprocess.STDOUT,
                text=True,
            )

            # First-run model downloads can take a while on Colab. Report log output
            # periodically rather than leaving a silent, apparently stalled cell.
            deadline = time.time() + 1800
            last_log_report = 0
            video_server_ready = False
            while time.time() < deadline:
                if video_server_proc.poll() is not None:
                    break
                try:
                    with urlopen(Request(VIDEO_HEALTH_URL), timeout=3) as response:
                        import json
                        health = json.loads(response.read().decode('utf-8'))
                    if health.get('status') == 'ready':
                        video_server_ready = True
                        break
                except (URLError, TimeoutError, ValueError):
                    pass

                if time.time() - last_log_report >= 30:
                    last_log_report = time.time()
                    print('AnimateDiff is downloading/loading its models...')
                    if VIDEO_LOG_PATH.is_file():
                        print(
                            VIDEO_LOG_PATH.read_text(
                                encoding='utf-8', errors='replace'
                            )[-1800:]
                        )
                time.sleep(5)

            if video_server_ready:
                set_video_source('animatediff')
                print(
                    f'Local AnimateDiff is ready on {gpu_name}. '
                    'Model weights are cached on Google Drive.'
                )
                print(
                    'MoneyPrinterTurbo now offers Local AnimateDiff (Free GPU) '
                    'as a video source.'
                )
            else:
                log_tail = (
                    VIDEO_LOG_PATH.read_text(
                        encoding='utf-8', errors='replace'
                    )[-6000:]
                    if VIDEO_LOG_PATH.is_file() else 'No server log was produced.'
                )
                use_existing_visual_source(
                    'the model did not become ready within 30 minutes'
                )
                print(log_tail)
    except Exception as exc:
        use_existing_visual_source(f'{type(exc).__name__}: {exc}')

## 5. Start the free Solo Stack voice service

This installs the CPU Kokoro-FastAPI service in its own isolated environment and pins upstream commit `5364fc31c770bd60069edc60521eb7bca4e5f5a9`. The model files are stored in Google Drive and their SHA-256 hashes are checked again on every runtime start. The notebook auditions both plain speech and the exact SSML pacing request used by Solo Stack before selecting the stable `am_michael` voice. If either audition or service startup fails, the existing voice configuration is preserved. The service binds to loopback and is not exposed through the ngrok tunnel. This is a CPU TTS service and may take several minutes to install and warm up.

In [ ]:
import os
import signal
import shutil
import hashlib
import subprocess
import time
from pathlib import Path
from urllib.request import urlopen, Request
import json

KOKORO_DIR = Path("/content/Kokoro-FastAPI")
KOKORO_ENV = KOKORO_DIR / ".venv"
KOKORO_LOG = Path("/content/kokoro-fastapi.log")
KOKORO_URL = "http://127.0.0.1:8880/v1/audio/voices"
KOKORO_REPO = "https://github.com/remsky/Kokoro-FastAPI.git"
KOKORO_COMMIT = "5364fc31c770bd60069edc60521eb7bca4e5f5a9"

# Pin the upstream server source so its API and CPU dependency set cannot drift
# between Colab sessions. The selected upstream commit also verifies the model
# files against published SHA-256 digests before loading them.
if not (KOKORO_DIR / ".git").is_dir():
    if KOKORO_DIR.exists():
        raise RuntimeError(f"{KOKORO_DIR} exists but is not a Git repository")
    subprocess.run([
        "git", "clone", "--depth", "1", KOKORO_REPO, str(KOKORO_DIR)
    ], check=True)
else:
    subprocess.run([
        "git", "-C", str(KOKORO_DIR), "remote", "set-url", "origin", KOKORO_REPO
    ], check=True)

subprocess.run([
    "git", "-C", str(KOKORO_DIR), "fetch", "--depth", "1",
    "origin", KOKORO_COMMIT
], check=True)
subprocess.run([
    "git", "-C", str(KOKORO_DIR), "checkout", "--force", "--detach",
    KOKORO_COMMIT
], check=True)

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "espeak-ng"], check=True)

MPT_HOME = Path(os.environ.get("MPT_HOME", str(REPO_DIR))).expanduser().resolve()
MPT_HOME.mkdir(parents=True, exist_ok=True)
MODEL_STORE = MPT_HOME / "model-cache" / "kokoro" / "v1_0"
MODEL_STORE.mkdir(parents=True, exist_ok=True)

# Kokoro-FastAPI's pinned downloader writes to api/src/models/v1_0. Point that
# exact path at Drive so the ~300 MB verified model survives Colab resets.
LOCAL_MODEL_DIR = KOKORO_DIR / "api" / "src" / "models" / "v1_0"
LOCAL_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
if LOCAL_MODEL_DIR.is_symlink():
    if LOCAL_MODEL_DIR.resolve() != MODEL_STORE.resolve():
        LOCAL_MODEL_DIR.unlink()
elif LOCAL_MODEL_DIR.exists():
    if not LOCAL_MODEL_DIR.is_dir():
        raise RuntimeError(f"Kokoro model path is not a directory: {LOCAL_MODEL_DIR}")
    shutil.copytree(LOCAL_MODEL_DIR, MODEL_STORE, dirs_exist_ok=True)
    shutil.rmtree(LOCAL_MODEL_DIR)
if not LOCAL_MODEL_DIR.exists():
    LOCAL_MODEL_DIR.symlink_to(MODEL_STORE, target_is_directory=True)

# Re-check persisted Drive files on every run. The upstream downloader verifies
# fresh downloads, but its existing-file fast path checks size/JSON rather than
# re-hashing a previously cached model.
EXPECTED_MODEL_HASHES = {
    "kokoro-v1_0.pth": "496dba118d1a58f5f3db2efc88dbdc216e0483fc89fe6e47ee1f2c53f18ad1e4",
    "config.json": "5abb01e2403b072bf03d04fde160443e209d7a0dad49a423be15196b9b43c17f",
}
cached_model_files = {
    name: MODEL_STORE / name for name in EXPECTED_MODEL_HASHES
}
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


if all(path.is_file() for path in cached_model_files.values()):
    invalid_cache = []
    for name, path in cached_model_files.items():
        if sha256_file(path) != EXPECTED_MODEL_HASHES[name]:
            invalid_cache.append(name)
    if invalid_cache:
        quarantine = MODEL_STORE / ("quarantine-" + time.strftime("%Y%m%d-%H%M%S"))
        quarantine.mkdir(parents=True, exist_ok=True)
        for path in cached_model_files.values():
            if path.exists():
                path.replace(quarantine / path.name)
        print(
            "Cached Kokoro model failed SHA-256 verification; preserved the old "
            f"files in {quarantine} and will download verified copies."
        )

if not KOKORO_ENV.exists():
    subprocess.run([
        "uv", "venv", "--python", "3.11", str(KOKORO_ENV)
    ], cwd=KOKORO_DIR, check=True)

kokoro_env = os.environ.copy()
kokoro_env["VIRTUAL_ENV"] = str(KOKORO_ENV)
kokoro_env["PATH"] = str(KOKORO_ENV / "bin") + os.pathsep + kokoro_env.get("PATH", "")
kokoro_env["UV_PROJECT_ENVIRONMENT"] = str(KOKORO_ENV)
kokoro_env["USE_GPU"] = "false"
kokoro_env["USE_ONNX"] = "false"
kokoro_env["HOST"] = "127.0.0.1"
kokoro_env["PORT"] = "8880"
kokoro_env["ENABLE_SSML"] = "true"
kokoro_env["ENABLE_VOICE_TAGS"] = "true"
kokoro_env["ESPEAK_DATA_PATH"] = "/usr/lib/x86_64-linux-gnu/espeak-ng-data"
kokoro_env["HF_HOME"] = str(MPT_HOME / "model-cache" / "huggingface")
Path(kokoro_env["HF_HOME"]).mkdir(parents=True, exist_ok=True)

old_proc = globals().get("kokoro_server_proc")
if old_proc is not None and old_proc.poll() is None:
    try:
        os.killpg(old_proc.pid, signal.SIGTERM)
        old_proc.wait(timeout=15)
    except ProcessLookupError:
        pass
    except subprocess.TimeoutExpired:
        os.killpg(old_proc.pid, signal.SIGKILL)
        old_proc.wait(timeout=10)
old_log_handle = globals().get("kokoro_log_handle")
if old_log_handle is not None and not old_log_handle.closed:
    old_log_handle.close()

kokoro_log_handle = KOKORO_LOG.open("w", encoding="utf-8")
kokoro_server_proc = subprocess.Popen(
    ["bash", "start-cpu.sh"],
    cwd=KOKORO_DIR,
    env=kokoro_env,
    stdout=kokoro_log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

voices_ready = False
last_error = None
for _ in range(180):
    if kokoro_server_proc.poll() is not None:
        break
    try:
        with urlopen(Request(KOKORO_URL), timeout=5) as response:
            payload = json.loads(response.read().decode("utf-8"))
        voices = payload.get("voices", []) if isinstance(payload, dict) else payload
        voice_ids = [
            voice.get("id") if isinstance(voice, dict) else voice
            for voice in voices
            if isinstance(voice, (dict, str))
        ]
        if "am_michael" in voice_ids:
            voices_ready = True
            break
    except Exception as exc:
        last_error = type(exc).__name__
    time.sleep(2)

if voices_ready:
    import requests
    import toml
    from IPython.display import Audio, display

    audition_text = (
        "Would you trust an AI tool with your next client proposal? "
        "We tested the real workflow, checked the limits, and looked at what it actually saves."
    )
    audition_response = requests.post(
        "http://127.0.0.1:8880/v1/audio/speech",
        json={
            "model": "kokoro",
            "input": audition_text,
            "voice": "am_michael",
            "response_format": "mp3",
            "speed": 1.0,
        },
        timeout=180,
    )
    audition_response.raise_for_status()
    audition_bytes = audition_response.content
    if len(audition_bytes) < 4096:
        raise RuntimeError("Kokoro voice audition returned an unexpectedly small audio file")

    # Verify the exact SSML/rate path used by Solo Stack before changing the
    # user's active TTS provider. A plain speech response alone is insufficient.
    ssml_response = requests.post(
        "http://127.0.0.1:8880/v1/audio/speech",
        json={
            "model": "kokoro",
            "input": (
                "<speak><prosody rate=\"95%\">"
                "Check the free plan carefully before you choose."
                "</prosody><break time=\"180ms\"/>"
                "Then compare the result with the work you actually need."
                "</speak>"
            ),
            "voice": "am_michael",
            "response_format": "mp3",
            "speed": 1.0,
            "ssml": True,
            "allow_voice_tags": True,
        },
        timeout=180,
    )
    ssml_response.raise_for_status()
    ssml_bytes = ssml_response.content
    if len(ssml_bytes) < 4096:
        raise RuntimeError("Kokoro SSML audition returned an unexpectedly small audio file")

    audition_path = MPT_HOME / "solo_stack_voice_audition.mp3"
    ssml_audition_path = MPT_HOME / "solo_stack_voice_ssml_audition.mp3"
    audition_path.write_bytes(audition_bytes)
    ssml_audition_path.write_bytes(ssml_bytes)

    config_path = MPT_HOME / "config.toml"
    if not config_path.is_file():
        raise RuntimeError(f"MoneyPrinterTurbo config is missing: {config_path}")
    config_data = toml.load(config_path)
    config_data.setdefault("kokoro", {}).update({
        "base_url": "http://127.0.0.1:8880/v1",
        "model_id": "kokoro",
        "voices": ["am_michael"],
        "solo_stack_performance_enabled": True,
    })
    config_data.setdefault("ui", {}).update({
        "tts_server": "kokoro",
        "voice_name": "kokoro:am_michael",
    })
    with config_path.open("w", encoding="utf-8") as handle:
        toml.dump(config_data, handle)

    print(
        "Pinned Kokoro server, verified model cache, plain speech and SSML "
        "auditions are ready. Listen to both samples before producing the first episode."
    )
    display(Audio(filename=str(audition_path)))
    display(Audio(filename=str(ssml_audition_path)))
else:
    tail = KOKORO_LOG.read_text(encoding="utf-8", errors="replace")[-5000:] if KOKORO_LOG.exists() else ""
    print(
        f"Kokoro did not become ready (last check: {last_error}). "
        "Existing voice settings were preserved."
    )
    print(tail)


## 6. Run the Solo Stack preflight suite

Before exposing the WebUI, this cell runs the focused local tests for the presenter, product-evidence pipeline, free-only provider guards, script audit, Colab setup and final-render quality gate. It makes no external model or paid API requests. The UI launch is stopped if any test fails.

In [ ]:
import os
import shutil
import subprocess
import tempfile
from pathlib import Path

REPO_DIR = Path('/content/MoneyPrinterTurbo')
PROJECT_PYTHON = REPO_DIR / '.venv' / 'bin' / 'python'
if not PROJECT_PYTHON.is_file():
    raise RuntimeError(
        "MoneyPrinterTurbo's isolated Python environment is missing. "
        "Re-run the installation cell before the Solo Stack preflight."
    )
test_targets = [
    'test/services/test_colab_notebook.py',
    'test/services/test_content_intelligence.py',
    'test/services/test_solo_stack_director.py',
    'test/services/test_solo_stack_foundation.py',
    'test/services/test_solo_stack_presenter_quality.py',
    'test/services/test_solo_stack_product_media.py',
    'test/services/test_solo_stack_visual_retrieval.py',
    'test/services/test_task.py',
    'test/services/test_video.py',
    'test/services/test_material.py',
    'tests/test_visual_quality.py',
]
with tempfile.TemporaryDirectory(prefix='solo-stack-preflight-') as test_home:
    test_home_path = Path(test_home)
    shutil.copy2(REPO_DIR / 'config.example.toml', test_home_path / 'config.toml')
    test_env = os.environ.copy()
    test_env['MPT_HOME'] = str(test_home_path)
    print('Running Solo Stack preflight tests with an isolated test config...')
    subprocess.run(
        [str(PROJECT_PYTHON), '-m', 'pytest', '-q', '--disable-warnings', *test_targets],
        cwd=REPO_DIR,
        env=test_env,
        check=True,
    )
print('Solo Stack preflight passed. WebUI launch is now permitted.')


## 7. Configure the Colab tunnel

MoneyPrinterTurbo itself does not require ngrok for normal local use. This notebook uses ngrok only because the Streamlit server runs inside a remote Colab runtime.

Create an account and copy your authentication token from the [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken). The next cell reads it without displaying or storing it in the notebook.

In [ ]:
from getpass import getpass

from pyngrok import ngrok

# Close tunnels from earlier runs before configuring a new one.
ngrok.kill()

# Add NGROK_AUTHTOKEN once in Colab's Secrets panel (key icon).
# The prompt remains as a fallback if the secret is not available.
ngrok_token = None
try:
    from google.colab import userdata
    ngrok_token = userdata.get("NGROK_AUTHTOKEN")
except Exception:
    pass

if not ngrok_token:
    ngrok_token = getpass("Enter your ngrok authentication token: ").strip()
if not ngrok_token:
    raise ValueError("An ngrok authentication token is required")
ngrok.set_auth_token(ngrok_token)
del ngrok_token
print("ngrok authentication configured")

## 8. Launch the WebUI

The cell waits for Streamlit to become healthy before creating the public URL. If startup fails, it includes the recent server log in the error. After opening the URL, use **Settings** to configure the required model and media API keys. To use OpenRouter free LLMs, select **OpenRouter**, enter your OpenRouter API key, and keep the default `minimax/minimax-m3:free` model or choose another model from https://openrouter.ai/models?variant=free.

In [ ]:
import time
from pathlib import Path
import subprocess
from urllib.error import URLError
from urllib.request import urlopen

from pyngrok import ngrok

# Sanity checks before starting
REPO_DIR = Path("/content/MoneyPrinterTurbo")
if not REPO_DIR.is_dir():
    raise RuntimeError(f"Repo directory does not exist: {REPO_DIR}")
if not (REPO_DIR / "webui" / "Main.py").is_file():
    raise RuntimeError(f"webui/Main.py not found in {REPO_DIR}")

PORT = 8501
LOG_PATH = Path("/content/moneyprinterturbo-webui.log")

# Make this cell safe to rerun by closing the previous process, log handle, and tunnel.
previous_streamlit_proc = globals().get("streamlit_proc")
if previous_streamlit_proc is not None and previous_streamlit_proc.poll() is None:
    previous_streamlit_proc.terminate()
    previous_streamlit_proc.wait(timeout=10)
previous_streamlit_log = globals().get("streamlit_log")
if previous_streamlit_log is not None and not previous_streamlit_log.closed:
    previous_streamlit_log.close()

try:
    ngrok.kill()
except Exception:
    pass

streamlit_log = LOG_PATH.open("w", encoding="utf-8")
streamlit_proc = subprocess.Popen(
    [
        "uv",
        "run",
        "streamlit",
        "run",
        "webui/Main.py",
        f"--server.port={PORT}",
        "--server.address=0.0.0.0",
        "--browser.gatherUsageStats=False",
        "--client.toolbarMode=minimal",
        "--server.showEmailPrompt=False",
    ],
    cwd=REPO_DIR,
    stdout=streamlit_log,
    stderr=subprocess.STDOUT,
    text=True,
)

# Poll the health endpoint because startup time varies across Colab runtimes.
deadline = time.time() + 90
server_ready = False
while time.time() < deadline:
    if streamlit_proc.poll() is not None:
        break
    try:
        with urlopen(f"http://127.0.0.1:{PORT}/_stcore/health", timeout=2) as response:
            server_ready = response.status == 200
    except (URLError, TimeoutError):
        pass
    if server_ready:
        break
    time.sleep(2)

if not server_ready:
    streamlit_log.flush()
    recent_log = LOG_PATH.read_text(encoding="utf-8", errors="replace")[-4000:]
    raise RuntimeError(f"Streamlit failed to start. Recent log:\n{recent_log}")

# Use an explicit IPv4 URL because pyngrok may otherwise route localhost through ::1.
public_tunnel = ngrok.connect(addr=f"http://127.0.0.1:{PORT}")
print("MoneyPrinterTurbo is ready:")
print(public_tunnel.public_url)
print(f"Server log: {LOG_PATH}")